# Test du modèle Wake Word `hey_zephyr` (ONNX)

Ce notebook charge le modèle ONNX exporté, puis calcule pour chaque fichier audio son **score maximal de détection** avec le pipeline `openWakeWord` (mél-spectrogramme → embedding → modèle ONNX).

- **Dans Google Colab** : exécutez les cellules dans l'ordre ; vous pourrez importer le modèle `.onnx`, puis plusieurs fichiers audio.
- **Dans Jupyter local / WSL2** : placez le modèle et les audios dans des dossiers accessibles, puis renseignez leurs chemins dans la cellule **Paramètres**. `ffmpeg` doit être installé et accessible dans le terminal.

**Important :** le score est compris entre 0 et 1, mais ce n'est pas une probabilité calibrée. Un score élevé indique une ressemblance avec le mot appris ; le seuil `0.5` est un repère initial à ajuster après essais positifs et négatifs.


In [ ]:
# Installation des dépendances Python
%pip install -q --upgrade openwakeword onnxruntime soundfile scipy scikit-learn "numpy<2.3" "pandas==2.2.3"


## 1. Paramètres


In [ ]:
import os
from pathlib import Path

# Seuil de lecture indicatif, pas une valeur universelle.
THRESHOLD = 0.50

# Utilisés uniquement en Jupyter local / WSL2.
# Exemple Windows : r"C:\Users\VotreNom\Downloads\hey_zephyr.onnx"
MODEL_PATH = r"./hey_zephyr.onnx"
AUDIO_DIR = r"./audios"  # dossier contenant les fichiers à tester

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Environnement :", "Google Colab" if IN_COLAB else "Jupyter local")
print("Seuil indicatif :", THRESHOLD)


## 2. Charger le modèle ONNX
Dans Colab, importez `hey_zephyr.onnx`. En local, le chemin `MODEL_PATH` défini plus haut doit pointer vers le fichier.


In [ ]:
from pathlib import Path

if IN_COLAB:
    uploaded_model = files.upload()
    model_candidates = [name for name in uploaded_model if name.lower().endswith(".onnx")]
    if not model_candidates:
        raise FileNotFoundError("Aucun fichier .onnx importé. Importez le modèle exporté.")
    MODEL_PATH = str(Path(model_candidates[0]).resolve())
else:
    MODEL_PATH = str(Path(MODEL_PATH).expanduser().resolve())

if not Path(MODEL_PATH).is_file():
    raise FileNotFoundError(
        f"Modèle introuvable : {MODEL_PATH}\n"
        "Corrigez MODEL_PATH ou placez hey_zephyr.onnx dans le dossier du notebook."
    )

print(f"Modèle : {MODEL_PATH}")
print(f"Taille : {Path(MODEL_PATH).stat().st_size / 1024:.1f} Ko")


## 3. Initialiser openWakeWord
On utilise `openWakeWord` pour effectuer le prétraitement audio attendu par le modèle personnalisé, et pas une inférence ONNX brute sur les échantillons audio.


In [ ]:
import numpy as np
from openwakeword.model import Model

oww = Model(
    wakeword_models=[MODEL_PATH],
    inference_framework="onnx",
)
KEY = next(iter(oww.models))
print("Modèle chargé.")
print("Clé de sortie utilisée :", KEY)

def clip_score(wav16_int16):
    """Retourne le score maximal sur tout le clip audio (0 à 1)."""
    oww.reset()
    predictions = oww.predict_clip(np.asarray(wav16_int16, dtype=np.int16))
    scores = [float(p[KEY]) for p in predictions if KEY in p]
    return max(scores) if scores else 0.0


## 4. Charger les pistes audio
Formats courants acceptés : WAV, MP3, M4A, OGG, FLAC… `ffmpeg` convertit chaque piste en mono, 16 kHz, comme dans le notebook d'entraînement.


In [ ]:
import subprocess
import tempfile
import soundfile as sf
import pandas as pd
from pathlib import Path

if IN_COLAB:
    uploaded_audio = files.upload()
    audio_paths = [
        Path(name).resolve()
        for name in uploaded_audio
        if not name.lower().endswith(".onnx")
    ]
else:
    audio_root = Path(AUDIO_DIR).expanduser()
    if not audio_root.exists():
        raise FileNotFoundError(
            f"Dossier audio introuvable : {audio_root.resolve()}\n"
            "Créez ce dossier, placez-y vos audios et vérifiez AUDIO_DIR."
        )
    supported = {".wav", ".mp3", ".m4a", ".ogg", ".flac", ".aac", ".wma", ".webm"}
    audio_paths = sorted(p.resolve() for p in audio_root.iterdir()
                         if p.is_file() and p.suffix.lower() in supported)

if not audio_paths:
    raise FileNotFoundError("Aucun fichier audio trouvé/importé.")

print(f"{len(audio_paths)} fichier(s) audio à tester :")
for p in audio_paths:
    print(" •", p.name)


## 5. Calculer les scores
Le tableau indique le score maximal obtenu sur chaque fichier, ainsi qu'un résultat indicatif selon le seuil choisi. Un score supérieur au seuil ne garantit pas à lui seul une détection fiable en situation réelle.


In [ ]:
results = []

for audio_path in audio_paths:
    try:
        with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
            converted_path = tmp.name

        subprocess.run(
            [
                "ffmpeg", "-y", "-loglevel", "error",
                "-i", str(audio_path),
                "-ac", "1", "-ar", "16000",
                converted_path,
            ],
            check=True,
            capture_output=True,
            text=True,
        )

        x, sr = sf.read(converted_path, dtype="float32")
        if x.ndim > 1:
            x = x.mean(axis=1)
        if sr != 16000:
            raise ValueError(f"Fréquence inattendue après conversion : {sr} Hz")
        if len(x) == 0:
            raise ValueError("Fichier audio vide.")

        audio_int16 = (np.clip(x, -1.0, 1.0) * 32767).astype(np.int16)
        score = clip_score(audio_int16)

        results.append({
            "fichier": audio_path.name,
            "durée_s": round(len(audio_int16) / 16000, 2),
            "score_max": round(score, 4),
            f">= {THRESHOLD:.2f} (indicatif)": score >= THRESHOLD,
            "statut": "OK",
            "erreur": "",
        })
        print(f"{audio_path.name:45s} score max = {score:.3f}")
    except Exception as exc:
        results.append({
            "fichier": audio_path.name,
            "durée_s": None,
            "score_max": None,
            f">= {THRESHOLD:.2f} (indicatif)": None,
            "statut": "ERREUR",
            "erreur": str(exc),
        })
        print(f"ERREUR — {audio_path.name}: {exc}")
    finally:
        if "converted_path" in locals() and os.path.exists(converted_path):
            os.remove(converted_path)
        converted_path = None

results_df = pd.DataFrame(results)
display(results_df)


## 6. Exporter les résultats (facultatif)
Le CSV permet de comparer les enregistrements positifs (où vous dites « Hey Zephyr ») et négatifs (paroles normales, mots proches, silence/bruit). Pour juger le modèle, testez plusieurs prises de voix et plusieurs environnements.


In [ ]:
csv_path = "hey_zephyr_scores.csv"
results_df.to_csv(csv_path, index=False, encoding="utf-8-sig")
print("CSV créé :", Path(csv_path).resolve())

if IN_COLAB:
    files.download(csv_path)


## Conseils pour interpréter les scores

- Testez plusieurs fichiers où la phrase est clairement prononcée, avec des distances et volumes différents.
- Ajoutez des fichiers négatifs : conversations ordinaires, « hey Siri », « hey Google », mots proches comme « zephyr » seul, ainsi que du bruit ambiant.
- Comparez les distributions de scores positifs et négatifs avant de fixer un seuil définitif. Le seuil `0.5` est uniquement indicatif.
- Le score maximal est sensible à la durée : un fichier long offre davantage d'occasions d'obtenir un pic. Comparez aussi des clips de durée similaire.
- Si l'import du modèle échoue, vérifiez que le fichier choisi est bien le modèle personnalisé exporté et que la version de `openwakeword` est compatible avec le modèle.
